# Training YOLOv8 Smart Gate

Notebook ini memakai `dataset_merged/` sebagai dataset utama. `plat-nomor/` digunakan sebagai fallback jika dataset gabungan tidak tersedia.

In [20]:
!pip install -q ultralytics pyyaml gdown

from pathlib import Path
import zipfile
import shutil
import gdown
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU CUDA tidak aktif. Di Colab pilih Runtime > Change runtime type > T4 GPU lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))

# Arsip training diambil berdasarkan file ID Drive agar unduhan stabil.
DRIVE_FILES = {
    'smartgate-dataset-merged.zip': '1mZxUtYrBLjsXVrzKvoZDhia9PZ9Y14AG',
}
SOURCE_ROOT = Path('/content/smartgate-sources')
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = []
for filename, file_id in DRIVE_FILES.items():
    destination = SOURCE_ROOT / filename
    if not destination.is_file() or destination.stat().st_size < 1024:
        gdown.download(id=file_id, output=str(destination), quiet=False)
    if not destination.is_file() or destination.stat().st_size < 1024:
        raise RuntimeError(f'Download gagal atau file terlalu kecil: {filename}')
    zip_files.append(destination)

# Ekstrak setiap ZIP ke folder terpisah agar sumber dan anotasi tidak tertimpa.
extracted = {}
for zip_path in zip_files:
    target = SOURCE_ROOT / zip_path.stem
    marker = target / '.extracted-ok'
    if not marker.exists():
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as archive:
            bad_member = archive.testzip()
            if bad_member is not None:
                raise RuntimeError(f'ZIP rusak pada {zip_path.name}, entry: {bad_member}')
            archive.extractall(target)
        marker.write_text('ok', encoding='utf-8')
    extracted[zip_path.name] = target

main_zip = SOURCE_ROOT / 'smartgate-dataset-merged.zip'
main_root = extracted[main_zip.name]
candidates = [p.parent for p in main_root.rglob('data.yaml') if (p.parent / 'train').is_dir()]
preferred = [p for p in candidates if p.name == 'plat-nomor']
DATA_ROOT = next(iter(preferred or candidates), None)
if DATA_ROOT is None:
    raise FileNotFoundError('data.yaml tidak ditemukan setelah ekstraksi smartgate-dataset-merged.zip.')
DATA_YAML = DATA_ROOT / 'data.yaml'
EXTRA_ZIP_FILES = [p for p in zip_files if p.name != main_zip.name]
EXTRA_EXTRACT_ROOT = SOURCE_ROOT
print('Drive ZIP files:')
for item in zip_files:
    print(f'  - {item.name}: {item.stat().st_size / 1024**2:.2f} MB')
print('Dataset utama:', DATA_ROOT)
print('Config sumber:', DATA_YAML)
print('ZIP tambahan:', len(EXTRA_ZIP_FILES))


GPU: Tesla T4
Drive ZIP files:
  - smartgate-dataset-merged.zip: 677.79 MB
Dataset utama: /content/smartgate-sources/smartgate-dataset-merged
Config sumber: /content/smartgate-sources/smartgate-dataset-merged/data.yaml
ZIP tambahan: 0


In [21]:
import yaml, hashlib, shutil, json

config = yaml.safe_load(DATA_YAML.read_text())
config['path'] = str(DATA_ROOT)
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'
config['nc'] = 1
config['names'] = ['number-plate']
RUNTIME_YAML = Path('/content/dataset_runtime.yaml')
RUNTIME_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(RUNTIME_YAML.read_text())

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
train_img_dir = DATA_ROOT / 'train' / 'images'
train_lbl_dir = DATA_ROOT / 'train' / 'labels'
val_test_stems = {p.stem for split in ('valid', 'test') for p in (DATA_ROOT / split / 'images').iterdir() if p.is_file()}

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

val_test_hashes = {file_hash(p) for split in ('valid', 'test') for p in (DATA_ROOT / split / 'images').iterdir() if p.is_file()}

summary = {'roboflow': 0, 'kaggle': 0, 'background': 0, 'skipped_leakage': 0, 'invalid': 0}

def valid_yolo_lines(raw_lines):
    out = []
    for line in raw_lines:
        line = line.strip()
        if not line:
            continue
        f = line.split()
        if len(f) != 5:
            return None
        try:
            cls = int(f[0]); vals = [float(x) for x in f[1:]]
        except ValueError:
            return None
        if cls != 0 or any(v < 0 or v > 1 for v in vals):
            return None
        out.append(line)
    return out

def add_image(img_path, lines, prefix, counter):
    if img_path.stem in val_test_stems or file_hash(img_path) in val_test_hashes:
        summary['skipped_leakage'] += 1
        return
    stem = f'{prefix}_{counter:05d}'
    shutil.copy2(img_path, train_img_dir / f'{stem}{img_path.suffix.lower()}')
    (train_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))

# --- Roboflow: hanya split train (valid/test utama tidak disentuh) ---
rf_root = SOURCE_ROOT / 'Indonesia License Plate Detection.v1i.yolov8'
if (rf_root / 'train' / 'images').is_dir():
    i = 0
    for img in sorted((rf_root / 'train' / 'images').iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        lbl = rf_root / 'train' / 'labels' / f'{img.stem}.txt'
        if not lbl.is_file():
            summary['invalid'] += 1
            continue
        lines = valid_yolo_lines(lbl.read_text().splitlines())
        if lines is None:
            summary['invalid'] += 1
            continue
        add_image(img, lines, 'rf', i); i += 1
        summary['roboflow'] += 1

# --- Kaggle: konversi COCO -> YOLO ---
kg_root = SOURCE_ROOT / 'Kaggle plate dataset'
coco_files = sorted(kg_root.rglob('*.json'))
coco_path = next((p for p in coco_files if any(k in p.name.lower() for k in ('instances', 'coco', 'annotation'))), coco_files[0] if coco_files else None)
if coco_path is not None:
    coco = json.loads(coco_path.read_text())
    images_meta = {im['id']: im for im in coco.get('images', [])}
    by_img = {}
    for a in coco.get('annotations', []):
        by_img.setdefault(a['image_id'], []).append(a)
    plate_cats = [c['id'] for c in coco.get('categories', []) if 'plate' in c['name'].lower()] or [c['id'] for c in coco.get('categories', [])]
    files = {Path(im['file_name']).name for im in images_meta.values()}
    img_dirs = [p for p in kg_root.rglob('images') if p.is_dir()]
    img_dir = next((d for d in img_dirs if any((d / fn).exists() for fn in list(files)[:20])), img_dirs[0] if img_dirs else None)
    if img_dir is not None:
        i = 0
        for mid, im in sorted(images_meta.items()):
            img_path = img_dir / Path(im['file_name']).name
            if not img_path.exists():
                summary['invalid'] += 1
                continue
            W, H = float(im.get('width', 0) or 0), float(im.get('height', 0) or 0)
            lines, ok = [], True
            for a in by_img.get(mid, []):
                if a.get('category_id') not in plate_cats:
                    continue
                x, y, w, h = (float(v) for v in a['bbox'])
                if W and H:
                    cx, cy, bw, bh = (x + w / 2) / W, (y + h / 2) / H, w / W, h / H
                else:
                    ok = False; break
                if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                    ok = False; break
                lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            if not ok or not lines:
                summary['invalid'] += 1
                continue
            add_image(img_path, lines, 'kg', i); i += 1
            summary['kaggle'] += 1

# --- Background no-plate: label kosong ke train saja ---
bg_root = SOURCE_ROOT / 'no-plate-backgrounds'
if (bg_root / 'images').is_dir():
    i = 0
    for img in sorted((bg_root / 'images').iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        add_image(img, [], 'bg', i); i += 1
        summary['background'] += 1

print('MERGE SUMMARY:', summary)

# --- OCR reference data (crop plat + label teks) ---
OCR_TRAIN_ROOT = Path('/content/ocr_reference')
plate_text = kg_root / 'plate_text_dataset'
if plate_text.is_dir() and not OCR_TRAIN_ROOT.exists():
    shutil.copytree(plate_text, OCR_TRAIN_ROOT)
    print('OCR reference:', OCR_TRAIN_ROOT)

# --- Validasi seluruh split (label kosong = background, diizinkan) ---
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / config[split]
    label_dir = DATA_ROOT / config[split].replace('/images', '/labels')
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f'{split}: struktur tidak ditemukan di {image_dir}')
    images = {p.stem for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS}
    labels = {p.stem for p in label_dir.glob('*.txt')}
    missing = sorted(images - labels)
    orphan = sorted(labels - images)
    if missing or orphan:
        raise ValueError(f'{split}: missing labels={len(missing)}, orphan labels={len(orphan)}')
    bg = box = 0
    for lp in label_dir.glob('*.txt'):
        nonempty = [l.strip() for l in lp.read_text().splitlines() if l.strip()]
        if not nonempty:
            bg += 1
            continue
        for ln, line in enumerate(nonempty, 1):
            f = line.split()
            if len(f) != 5 or int(f[0]) != 0 or not all(0.0 <= float(v) <= 1.0 for v in f[1:]):
                raise ValueError(f'Anotasi invalid: {lp}:{ln} -> {line}')
            box += 1
    print(f'{split}: {len(images)} images ({bg} background, {box} kotak), valid')
print('Struktur dan anotasi dataset valid.')


train: train/images
val: valid/images
test: test/images
nc: 1
names:
- number-plate
path: /content/smartgate-sources/smartgate-dataset-merged

MERGE SUMMARY: {'roboflow': 0, 'kaggle': 0, 'background': 0, 'skipped_leakage': 0, 'invalid': 0}
train: 3982 images (17 background, 5154 kotak), valid
val: 84 images (0 background, 99 kotak), valid
test: 84 images (0 background, 102 kotak), valid
Struktur dan anotasi dataset valid.


In [22]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    files, objects = count_labels(label_dir)
    print(f'{split}: {files} file label, {objects} anotasi')

train: 3982 file label, 5154 anotasi
valid: 84 file label, 99 anotasi
test: 84 file label, 102 anotasi


In [23]:
# === 5-Fold Stratified Group Cross-Validation (dHash 64-bit + Multi-Domain Stratification) ===
# Mengelompokkan frame kendaraan berurutan dengan perceptual hash (dHash) agar tidak bocor
# antar-fold, dan membagi ~4.066 citra (train + valid) menjadi 5 fold (~3.253 train : ~813 val per fold)
# sementara split `test` tetap steril sebagai hold-out test set.
import cv2
import hashlib
import json

K_FOLDS = 5
FOLD_INDEX = 0          # Fold aktif bila RUN_ALL_FOLDS = False
RUN_ALL_FOLDS = False   # Set True di Colab untuk melatih & mengevaluasi seluruh 5 fold (mAP50_CV = mean +/- std)
HAMMING_THRESHOLD = 4

def _compute_dhash(img_gray, hash_size=8):
    if img_gray is None or img_gray.size == 0:
        return 0
    resized = cv2.resize(img_gray, (hash_size + 1, hash_size), interpolation=cv2.INTER_AREA)
    diff = resized[:, 1:] > resized[:, :-1]
    val = 0
    for bit in diff.flatten():
        val = (val << 1) | int(bool(bit))
    return val

def _infer_domain(stem):
    s = stem.lower()
    for p in ('rf_', 'kg_', 'mg_', 'bg_', 'web_'):
        if s.startswith(p):
            return p.rstrip('_')
    return 'orig'

def _classify_boxes(lines):
    areas, aspects = [], []
    for raw in lines:
        parts = raw.strip().split()
        if len(parts) == 5:
            _, _, _, bw, bh = (float(v) for v in parts)
            if bw > 0 and bh > 0:
                areas.append(bw * bh); aspects.append(bw / bh)
    if not areas:
        return 'bg_empty', 'none'
    mx = max(areas)
    scale_b = 'small' if mx < 0.015 else ('medium' if mx <= 0.050 else 'large')
    aspect_b = 'two_row' if (sum(aspects) / len(aspects)) < 2.0 else 'one_row'
    return scale_b, aspect_b

pool_items = []
for split in ('train', 'valid'):
    img_dir = DATA_ROOT / split / 'images'
    lbl_dir = DATA_ROOT / split / 'labels'
    for img_path in sorted(img_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in IMG_EXTS:
            continue
        rel_path = img_path.relative_to(DATA_ROOT).as_posix()
        lbl_path = lbl_dir / f'{img_path.stem}.txt'
        lines = lbl_path.read_text().splitlines() if lbl_path.is_file() else []
        dom = _infer_domain(img_path.stem)
        sb, ab = _classify_boxes(lines)
        stratum = f'{dom}__{sb}__{ab}'
        sha = file_hash(img_path)
        dh = _compute_dhash(cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE))
        pool_items.append((rel_path, dom, sb, stratum, dh, sha))

n_pool = len(pool_items)
parent = list(range(n_pool))
def _find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

by_sha = {}
for idx, it in enumerate(pool_items):
    if it[5] in by_sha:
        _union(idx, by_sha[it[5]])
    else:
        by_sha[it[5]] = idx
order = sorted(range(n_pool), key=lambda i: pool_items[i][4])
for pos, idx in enumerate(order):
    ha = pool_items[idx][4]
    if not ha:
        continue
    for nxt in order[pos + 1 : pos + 25]:
        hb = pool_items[nxt][4]
        if hb and (ha ^ hb).bit_count() <= HAMMING_THRESHOLD:
            _union(idx, nxt)

group_members = {}
for idx in range(n_pool):
    group_members.setdefault(_find(idx), []).append(idx)

group_info = []
for gid, members in group_members.items():
    st = pool_items[members[0]][3]
    th = int(hashlib.sha1(f'42:{gid}:{st}'.encode()).hexdigest()[:8], 16)
    group_info.append((st, -len(members), th, members))
group_info.sort()

fold_vals = [[] for _ in range(K_FOLDS)]
fold_totals = [0] * K_FOLDS
fold_st_counts = [{} for _ in range(K_FOLDS)]
for st, _, _, members in group_info:
    bf = min(range(K_FOLDS), key=lambda f: (fold_st_counts[f].get(st, 0), fold_totals[f], f))
    fold_vals[bf].extend(members)
    fold_totals[bf] += len(members)
    fold_st_counts[bf][st] = fold_st_counts[bf].get(st, 0) + len(members)

FOLDS_DIR = DATA_ROOT / 'folds'
FOLDS_DIR.mkdir(parents=True, exist_ok=True)
for c in FOLDS_DIR.glob('*.cache'):
    c.unlink()
FOLD_YAMLS = []
all_idx_set = set(range(n_pool))
for f_idx in range(K_FOLDS):
    v_set = set(fold_vals[f_idx])
    v_list = sorted(v_set)
    t_list = sorted(all_idx_set - v_set)
    (FOLDS_DIR / f'fold_{f_idx}_train.txt').write_text('\n'.join(str((DATA_ROOT / pool_items[i][0]).resolve()) for i in t_list) + '\n')
    (FOLDS_DIR / f'fold_{f_idx}_val.txt').write_text('\n'.join(str((DATA_ROOT / pool_items[i][0]).resolve()) for i in v_list) + '\n')
    f_yaml = FOLDS_DIR / f'data_fold_{f_idx}.yaml'
    f_yaml.write_text(yaml.safe_dump({
        'path': str(DATA_ROOT),
        'train': f'folds/fold_{f_idx}_train.txt',
        'val': f'folds/fold_{f_idx}_val.txt',
        'test': 'test/images',
        'nc': 1,
        'names': ['number-plate'],
    }, sort_keys=False))
    FOLD_YAMLS.append(f_yaml)
    dom_counts = {}
    for i in v_list:
        dom_counts[pool_items[i][1]] = dom_counts.get(pool_items[i][1], 0) + 1
    print(f'Fold {f_idx}: train={len(t_list)} val={len(v_list)} val_domains={dom_counts}')

RUNTIME_YAML = FOLD_YAMLS[FOLD_INDEX]
print(f'\nUnique visual groups (dHash anti-leakage): {len(group_members)} dari {n_pool} gambar.')
print('Active RUNTIME_YAML:', RUNTIME_YAML)


Fold 0: train=3259 val=807 val_domains={'bg': 3, 'kg': 39, 'mg': 231, 'orig': 95, 'rf': 439}
Fold 1: train=3259 val=807 val_domains={'bg': 2, 'kg': 39, 'mg': 230, 'orig': 96, 'rf': 440}
Fold 2: train=3226 val=840 val_domains={'bg': 3, 'kg': 44, 'mg': 263, 'orig': 94, 'rf': 436}
Fold 3: train=3260 val=806 val_domains={'bg': 2, 'kg': 40, 'mg': 230, 'orig': 96, 'rf': 438}
Fold 4: train=3260 val=806 val_domains={'bg': 2, 'kg': 44, 'mg': 225, 'orig': 96, 'rf': 439}

Unique visual groups (dHash anti-leakage): 3351 dari 4066 gambar.
Active RUNTIME_YAML: /content/smartgate-sources/smartgate-dataset-merged/folds/data_fold_0.yaml


## Training

Gunakan GPU Colab: **Runtime > Change runtime type > T4 GPU**. Kotak anotasi harus tepat mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.

In [24]:
from ultralytics import YOLO
import numpy as np
import torch

device = 0
batch = 16
BASE_MODEL = 'yolov8n.pt'
MODEL_TAG = BASE_MODEL.replace('.pt', '')
EXPERIMENT_NAME = f'smartgate-colab-v6-{MODEL_TAG}'
print('Device:', device)
print('Eksperimen utama:', EXPERIMENT_NAME)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
TARGET_MAP50 = 0.98

# Hiperparameter Generalisasi & Anti-Degradasi Spesifik Kamera Gate (TRAIN-2, TRAIN-4)
ANTI_DEGRADATION_HYPS = dict(
    cos_lr=True,
    lrf=0.01,
    warmup_epochs=3.0,
    weight_decay=0.001,
    dropout=0.10,
    box=7.5,
    # Geometrik realistis tanpa horizontal flip (fliplr WAJIB 0.0 agar urutan huruf plat tidak terbalik!)
    degrees=7.0,
    translate=0.10,
    scale=0.45,
    shear=2.5,
    perspective=0.0004,
    fliplr=0.0,
    flipud=0.0,
    # Oklusi & pencahayaan kamera gate / pantulan layar
    mosaic=0.8,
    mixup=0.05,
    copy_paste=0.10,
    erasing=0.25,
    close_mosaic=15,
    hsv_h=0.015,
    hsv_s=0.65,
    hsv_v=0.45,
)

def train_one_fold(fold_yaml, run_name):
    m = YOLO(BASE_MODEL)
    def stop_at_target(trainer):
        map50 = (trainer.metrics or {}).get('metrics/mAP50(B)', 0.0)
        if map50 >= TARGET_MAP50:
            trainer.stop = True
            print(f'\nTarget mAP50 {TARGET_MAP50:.2f} tercapai ({map50:.4f}) - training dihentikan.')
    m.add_callback('on_fit_epoch_end', stop_at_target)
    m.train(
        data=str(fold_yaml),
        epochs=50,
        imgsz=640,
        batch=batch,
        device=device,
        project=str(RUN_DIR),
        name=run_name,
        patience=15,
        pretrained=True,
        plots=True,
        val=True,
        seed=42,
        workers=2,
        cache=False,
        exist_ok=True,
        **ANTI_DEGRADATION_HYPS,
    )
    best_w = RUN_DIR / run_name / 'weights' / 'best.pt'
    val_m = YOLO(str(best_w)).val(data=str(fold_yaml), split='val', imgsz=640, device=device)
    return best_w, float(val_m.box.map50), float(val_m.box.map)

cv_fold_results = []
if RUN_ALL_FOLDS:
    best_cv_map50 = -1.0
    for f_idx, f_yaml in enumerate(FOLD_YAMLS):
        f_name = EXPERIMENT_NAME if f_idx == FOLD_INDEX else f'{EXPERIMENT_NAME}-fold{f_idx}'
        w_path, f_map50, f_map95 = train_one_fold(f_yaml, f_name)
        cv_fold_results.append({'fold': f_idx, 'mAP50': round(f_map50, 4), 'mAP50_95': round(f_map95, 4), 'weights': str(w_path)})
        if f_map50 > best_cv_map50:
            best_cv_map50 = f_map50
            RUNTIME_YAML = f_yaml
            if w_path != (RUN_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt'):
                (RUN_DIR / EXPERIMENT_NAME / 'weights').mkdir(parents=True, exist_ok=True)
                shutil.copy2(w_path, RUN_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt')
    map50_list = [r['mAP50'] for r in cv_fold_results]
    print(f'\n=== 5-FOLD CROSS-VALIDATION SUMMARY ({MODEL_TAG}) ===')
    print(f'mAP50_CV    : {np.mean(map50_list):.4f} +/- {np.std(map50_list):.4f}')
else:
    w_path, f_map50, f_map95 = train_one_fold(RUNTIME_YAML, EXPERIMENT_NAME)
    cv_fold_results.append({'fold': FOLD_INDEX, 'mAP50': round(f_map50, 4), 'mAP50_95': round(f_map95, 4), 'weights': str(w_path)})
print('Training selesai.')


Device: 0
Eksperimen utama: smartgate-colab-v6-yolov8n
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/smartgate-sources/smartgate-dataset-merged/folds/data_fold_0.yaml, degrees=7.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=50, erasing=0.25, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.65, hsv_v=0.45, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8n.pt, moment

In [25]:
best_pt = RUN_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
val_metrics = trained.val(data=str(RUNTIME_YAML), split='val', imgsz=640, device=device)
test_metrics = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device)
if float(test_metrics.box.map50) < 0.98:
    print('Menjalankan evaluasi test dengan TTA...')
    test_metrics_tta = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=640, device=device, augment=True)
    if float(test_metrics_tta.box.map50) > float(test_metrics.box.map50):
        test_metrics = test_metrics_tta
print('best.pt:', best_pt)
print(f'Validation mAP50: {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Test mAP50: {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95: {test_metrics.box.map:.4f}')

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2472.1±1637.3 MB/s, size: 191.5 KB)
val: Scanning /content/smartgate-sources/smartgate-dataset-merged/train/labels.cache... 807 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 807/807 211.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 8.0it/s 6.4s0.1ss
                   all        807       1188      0.866      0.798      0.849      0.567
Speed: 0.6ms preprocess, 3.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/runs/detect/val-6
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4199.0±1278.0 MB/s, s

In [26]:
print('Bobot selesai:', best_pt)
print('File model tersedia:', best_pt.is_file())

Bobot selesai: /content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.pt
File model tersedia: True


In [27]:
!pip install -q easyocr opencv-python-headless onnx onnxruntime

import cv2
import re
import time
import json
import torch
import numpy as np
import easyocr

ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
DIGIT_TO_CHAR = {'0': 'O', '1': 'I', '2': 'Z', '3': 'B', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B', '9': 'Q'}
CHAR_TO_DIGIT = {'O': '0', 'D': '0', 'Q': '0', 'I': '1', 'L': '1', 'Z': '2', 'A': '4', 'S': '5', 'G': '6', 'T': '7', 'B': '8'}
# V5 Perpol No. 7 Tahun 2021: kode wilayah 1-2 huruf, jangan ubah 'L' (Surabaya/Aceh) jadi 'D'
PREFIX_FIX = {'0': 'D', '1': 'D', 'O': 'D', 'Q': 'D', 'I': 'D', '8': 'B'}
VALID_PREFIXES = {'A','B','D','E','F','G','H','K','L','M','N','P','R','S','T','W','Z','AA','AB','AD','AE','AG','BA','BB','BD','BE','BG','BH','BK','BL','BM','BN','BP','DA','DB','DC','DD','DE','DG','DH','DK','DL','DM','DN','DP','DR','DT','DW','EA','EB','ED','KB','KH','KT','KU','PA','PB','PG','PS','PT','PY'}
DATE_LIKE = re.compile(r'^(?:(0[1-9]|1[0-2])[.\-·\s]?\d{2}|\d{1,2}[.\-·\s]\d{2,4})$')

def assess_crop_quality(crop_bgr):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(np.mean(gray))
    return {'blur_score': round(blur_score, 2), 'brightness': round(brightness, 2), 'is_usable': blur_score >= 30.0 and 20.0 <= brightness <= 245.0}

def detect_plate_color(crop_bgr):
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    center = hsv[int(h * .20):int(h * .75), int(w * .15):int(w * .85)]
    total = center.shape[0] * center.shape[1] + 1e-5
    yellow = cv2.inRange(center, (15, 80, 80), (35, 255, 255))
    red = cv2.bitwise_or(cv2.inRange(center, (0, 80, 80), (10, 255, 255)), cv2.inRange(center, (160, 80, 80), (180, 255, 255)))
    if np.sum(yellow > 0) / total > .25:
        return 'YELLOW'
    if np.sum(red > 0) / total > .25:
        return 'RED'
    return 'WHITE' if float(np.mean(center[:, :, 2])) > 127 else 'BLACK'

def preprocess_plate_for_ocr(crop_bgr, plate_color):
    h, w = crop_bgr.shape[:2]
    x_trim = int(w * .03)
    y_top = int(h * .04)
    y_bot = max(y_top + 10, int(h * .70))
    zone = crop_bgr[y_top:y_bot, x_trim:max(x_trim + 10, w - x_trim)]
    mh, mw = zone.shape[:2]
    if mw < 260:
        zone = cv2.resize(zone, None, fx=260.0 / max(mw, 1), fy=260.0 / max(mw, 1), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
    if plate_color == 'BLACK' or float(np.mean(gray)) < 110:
        gray = cv2.bitwise_not(gray)
    # V4: CLAHE lembut agar garis moire/pantulan dan baut tidak menonjol jadi karakter
    enhanced = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8)).apply(gray)
    return cv2.copyMakeBorder(enhanced, 6, 6, 8, 8, cv2.BORDER_REPLICATE)

def extract_main_ocr_text(ocr_out, img_h):
    if not ocr_out:
        return '', 0.0
    info = []
    for bbox, text, conf in ocr_out:
        raw_str = text.strip().upper()
        clean = re.sub(r'[^A-Z0-9]', '', raw_str)
        if not clean or DATE_LIKE.match(raw_str) or DATE_LIKE.match(clean):
            continue
        ys = [pt[1] for pt in bbox]
        xs = [pt[0] for pt in bbox]
        info.append({'x_min': min(xs), 'y_center': float(np.mean(ys)), 'height': max(ys) - min(ys), 'text': clean, 'conf': float(conf)})
    if not info:
        return '', 0.0
    # V4: dominan-line filter - buang baris teks pajak di bawah plat (y >= 0.78h)
    main = [it for it in info if it['y_center'] <= .78 * img_h] or info
    max_h = max(it['height'] for it in main)
    main = [it for it in main if it['height'] >= .45 * max_h] or main
    main.sort(key=lambda it: it['x_min'])
    return ''.join(it['text'] for it in main), float(np.mean([it['conf'] for it in main]))

def format_plate(raw_text):
    clean = re.sub(r'[^A-Z0-9]', '', raw_text.upper())
    if len(clean) >= 8 and DATE_LIKE.match(clean[-4:]):
        clean = clean[:-4]
    if len(clean) < 3:
        return clean, False
    # V4: buang awalan I/1/L/0/O yang terdeteksi sebelum kode wilayah
    if len(clean) >= 6 and clean[0] in ('I', '1', 'L', '0', 'O'):
        candidate = ''.join(PREFIX_FIX.get(c, c) for c in clean[1:3])
        if candidate in VALID_PREFIXES:
            clean = clean[1:]
    best = None; best_score = -999
    for start in range(min(2, len(clean))):
        for length in range(min(9, len(clean) - start), 2, -1):
            sub = clean[start:start + length]
            for p_len in (2, 1):
                if len(sub) <= p_len:
                    continue
                for d_len in range(min(4, len(sub) - p_len), 0, -1):
                    s_len = len(sub) - p_len - d_len
                    if s_len < 0 or s_len > 3:
                        continue
                    p_raw = sub[:p_len]; d_raw = sub[p_len:p_len + d_len]; s_raw = sub[p_len + d_len:]
                    if s_raw and s_raw.isdigit():
                        continue
                    p = p_raw if p_raw in VALID_PREFIXES else ''.join(PREFIX_FIX.get(c, c) for c in p_raw)
                    d = ''.join(CHAR_TO_DIGIT.get(c, c) for c in d_raw)
                    s = ''.join(DIGIT_TO_CHAR.get(c, c) for c in s_raw)
                    if not (p.isalpha() and d.isdigit() and (not s or s.isalpha())) or d[0] == '0' or p not in VALID_PREFIXES:
                        continue
                    score = (sum(4 if a == b else 1 for a, b in zip(p_raw, p)) + sum(5 if a == b else 1 for a, b in zip(d_raw, d)) + sum(4 if a == b else -3 for a, b in zip(s_raw, s)) - start * 8 - (len(clean) - start - length) * 4)
                    if 2 <= d_len <= 4 and 1 <= s_len <= 3 and any(c.isalpha() for c in s_raw):
                        score += 12
                    if score > best_score:
                        best_score = score; best = f'{p} {d} {s}'.strip()
    return (best, True) if best is not None and best_score >= 22 else (clean, False)

def plate_passes(crop_bgr):
    # V4 multi-pass: crop penuh + crop dengan pinggir lebih banyak dibuang (kisi bingkai)
    h, w = crop_bgr.shape[:2]
    passes = [crop_bgr]
    x = int(w * .08)
    if w - 2 * x > 40:
        passes.append(crop_bgr[:, x:w - x])
    return passes

def read_smartgate_plate(image_bgr, yolo_model, conf_thres=.35):
    started = time.perf_counter()
    result = yolo_model.predict(image_bgr, conf=conf_thres, imgsz=640, verbose=False)[0]
    detections = []
    h_img, w_img = image_bgr.shape[:2]
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        det_conf = float(box.conf[0])
        crop = image_bgr[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
        if crop.size == 0:
            continue
        quality = assess_crop_quality(crop)
        reads = []
        for variant in plate_passes(crop):
            if variant.size == 0:
                continue
            color = detect_plate_color(variant)
            processed = preprocess_plate_for_ocr(variant, color)
            ocr_out = ocr_reader.readtext(processed, allowlist=ALLOWLIST, detail=1, paragraph=False)
            raw, ocr_conf = extract_main_ocr_text(ocr_out, processed.shape[0])
            plate, valid = format_plate(raw)
            reads.append({'plate': plate, 'valid': valid, 'conf': ocr_conf, 'raw': raw, 'color': color})
        if not reads:
            continue
        valids = [r for r in reads if r['valid']]
        best = max(valids, key=lambda r: r['conf']) if valids else max(reads, key=lambda r: r['conf'])
        detections.append({'bbox': [x1, y1, x2, y2], 'plate_number': best['plate'], 'raw_ocr': best['raw'], 'is_valid_format': best['valid'], 'plate_color': best['color'], 'det_conf': round(det_conf, 4), 'ocr_conf': round(best['conf'], 4), 'quality': quality})
    return {'latency_ms': round((time.perf_counter() - started) * 1000, 2), 'plates': detections}

test_image_dir = DATA_ROOT / 'test' / 'images'
test_images = sorted(p for p in test_image_dir.iterdir() if p.is_file())
latencies = []; valid_reads = 0; detected = 0; sample_outputs = []
for img_path in test_images:
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    result = read_smartgate_plate(image, trained)
    latencies.append(result['latency_ms'])
    detected += len(result['plates'])
    valid_reads += sum(1 for item in result['plates'] if item['is_valid_format'])
    if len(sample_outputs) < 10 and result['plates']:
        sample_outputs.append({'file': img_path.name, **result['plates'][0]})
onnx_path = trained.export(format='onnx', imgsz=640, simplify=True, dynamic=False)
test_map50 = float(test_metrics.box.map50)
cv_folds_out = cv_fold_results if 'cv_fold_results' in globals() else []
cv_map50_mean = round(float(np.mean([r['mAP50'] for r in cv_folds_out])), 4) if cv_folds_out else None
cv_map50_std = round(float(np.std([r['mAP50'] for r in cv_folds_out])), 4) if cv_folds_out else None
report = {'model': str(best_pt), 'onnx_model': str(onnx_path), 'cv_folds': cv_folds_out, 'cv_mAP50_mean': cv_map50_mean, 'cv_mAP50_std': cv_map50_std, 'validation': {'mAP50': float(val_metrics.box.map50), 'mAP50_95': float(val_metrics.box.map)}, 'test': {'images': len(test_images), 'mAP50': test_map50, 'mAP50_95': float(test_metrics.box.map)}, 'ocr_and_gate_benchmark': {'total_detected_plates': detected, 'valid_indonesian_format_count': valid_reads, 'valid_format_rate': round(valid_reads / max(detected, 1), 4), 'avg_end_to_end_latency_ms': round(float(np.mean(latencies)), 2) if latencies else 0.0, 'p95_end_to_end_latency_ms': round(float(np.percentile(latencies, 95)), 2) if latencies else 0.0}, 'detector_target_mAP50': .98, 'detector_target_met': test_map50 >= .98, 'production_kpi_status': 'NOT_CLAIMED', 'sample_reads': sample_outputs}
print('=== LAPORAN EVALUASI DETECTOR & OCR SMART GATE (V4 PARSER) ===')
print(f'Validation mAP50    : {report["validation"]["mAP50"]:.4f}')
print(f'Validation mAP50-95 : {report["validation"]["mAP50_95"]:.4f}')
print(f'Test mAP50          : {report["test"]["mAP50"]:.4f}')
print(f'Test mAP50-95       : {report["test"]["mAP50_95"]:.4f}')
print(f'Valid Format Rate   : {report["ocr_and_gate_benchmark"]["valid_format_rate"]:.2%} ({valid_reads}/{detected})')
print(f'Avg latency         : {report["ocr_and_gate_benchmark"]["avg_end_to_end_latency_ms"]} ms')
print(f'P95 latency         : {report["ocr_and_gate_benchmark"]["p95_end_to_end_latency_ms"]} ms')
print(f'Detector target 98% : {"TERCAPAI" if report["detector_target_met"] else "BELUM TERCAPAI"}')
summary_path = RUN_DIR / EXPERIMENT_NAME / 'evaluation_summary.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(report, indent=2))
print('Ringkasan tersimpan:', summary_path)


Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from '/content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 0.9s, saved as '/content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.onnx' (12.2 MB)

Export complete (1.0s)
Results saved to /content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.onnx
Predict:         yolo predict task=detect model=/content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/smartgate-sources/runs/detect/smartgate-colab-v6-yolov8n/weights/best.onnx imgsz=640 data=/content

In [28]:
import os
import shutil
from IPython.display import FileLink, display

ARTIFACT_DIR = RUN_DIR / EXPERIMENT_NAME
REPORT_PATH = ARTIFACT_DIR / 'evaluation_summary.json'
required_files = [best_pt, REPORT_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Artefak belum tersedia: ' + ', '.join(missing))

DOWNLOAD_DIR = Path('/content/smartgate-artifacts')
DOWNLOAD_DIR.mkdir(exist_ok=True)
shutil.copy2(best_pt, DOWNLOAD_DIR / 'best.pt')
shutil.copy2(REPORT_PATH, DOWNLOAD_DIR / 'evaluation_summary.json')
for result_name in ('results.csv', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'best.onnx'):
    result_path = (Path(onnx_path) if result_name == 'best.onnx' else ARTIFACT_DIR / result_name)
    if result_path.is_file():
        shutil.copy2(result_path, DOWNLOAD_DIR / result_name)

# Simpan di working directory kernel agar dapat diambil oleh VS Code/Jupyter.
ARCHIVE_PATH = Path.cwd() / 'smartgate-artifacts-Y8N.zip'
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', DOWNLOAD_DIR)
if not ARCHIVE_PATH.is_file() or ARCHIVE_PATH.stat().st_size == 0:
    raise RuntimeError(f'ZIP artefak gagal dibuat: {ARCHIVE_PATH}')
print('Artefak siap diunduh:', ARCHIVE_PATH)
print(f'Ukuran ZIP: {ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MB')
print('File ZIP lokal kernel:', ARCHIVE_PATH)
display(FileLink(ARCHIVE_PATH.name, result_html_prefix='Download artefak: '))

# Runtime Colab dari VS Code tidak dapat mengirim FileLink langsung ke Windows.
# Salin artefak ke Google Drive agar dapat diambil dari drive.google.com.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/SmartGateSystem')
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_ARCHIVE_PATH = DRIVE_ARTIFACT_DIR / ARCHIVE_PATH.name
    shutil.copy2(ARCHIVE_PATH, DRIVE_ARCHIVE_PATH)
    if not DRIVE_ARCHIVE_PATH.is_file() or DRIVE_ARCHIVE_PATH.stat().st_size != ARCHIVE_PATH.stat().st_size:
        raise IOError(f'Verifikasi salinan Drive gagal: {DRIVE_ARCHIVE_PATH}')
    print('Artefak tersalin ke Google Drive:', DRIVE_ARCHIVE_PATH)
    print('Buka drive.google.com > My Drive > SmartGateSystem untuk mengambil ZIP.')
except (ImportError, OSError, IOError) as exc:
    print('Salinan Google Drive gagal:', exc)

# Download JavaScript hanya dipanggil pada web Google Colab.
if os.environ.get('COLAB_RELEASE_TAG'):
    from google.colab import files
    files.download(str(ARCHIVE_PATH))
else:
    print('Mode VS Code/Jupyter terdeteksi; gunakan tautan FileLink di atas.')


Artefak siap diunduh: /content/smartgate-artifacts-Y8N.zip
Ukuran ZIP: 16.18 MB
File ZIP lokal kernel: /content/smartgate-artifacts-Y8N.zip


/content/smartgate-artifacts-Y8N.zip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artefak tersalin ke Google Drive: /content/drive/MyDrive/SmartGateSystem/smartgate-artifacts-Y8N.zip
Buka drive.google.com > My Drive > SmartGateSystem untuk mengambil ZIP.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Setelah training selesai, salin `best.pt` ke:

`Tugas Citra Digital/prototype/runs/detect/smartgate-colab-v2/weights/best.pt`

Lalu jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut dan restart Flask.